# Train YOLO Pose Model (Bee Dataset)

This notebook trains a YOLO pose estimation model using data in `./labeled-data` (YOLO v1.0 pose labels).

## 1) Install/Import Dependencies

In [ ]:
# If needed, uncomment to install Ultralytics
# %pip install -q ultralytics pyyaml

import os
from pathlib import Path

from ultralytics import YOLO

## 2) Configure Paths

In [ ]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "labeled-data"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
DATA_YAML = DATA_DIR / "data.yaml"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_OUTPUT = ROOT / "ml_runs" / "pose"
PRETRAINED_WEIGHTS = MODEL_OUTPUT / "bee_pose" / "weights" / "best.pt"


assert DATA_DIR.exists(), f"Missing dataset folder: {DATA_DIR}"
assert IMAGES_DIR.exists(), f"Missing images folder: {IMAGES_DIR}"
assert LABELS_DIR.exists(), f"Missing labels folder: {LABELS_DIR}"
assert DATA_YAML.exists(), f"Missing YAML file: {DATA_YAML}"
assert PRETRAINED_WEIGHTS.exists(), f"Missing pretrained weights: {PRETRAINED_WEIGHTS}"

print(f"Workspace root: {ROOT}")
print(f"Dataset folder: {DATA_DIR}")

## 3a) Remove images without labels

In [ ]:
# Remove any images that don't have corresponding labels
for img_file in IMAGES_DIR.glob("*.png"):
    label_file = LABELS_DIR / img_file.with_suffix(".txt").name
    if not label_file.exists():
        print(f"Removing image without label: {img_file}")
        img_file.unlink()

## 3b) Redo autosplit if required

In [ ]:
# Autosplit the labeled-data into train, test, and validation sets
from ultralytics.data.split import autosplit


def redo_autosplit():
    # Ensure that the autosplit is run in the labeled-data directory, but return to the base directory
    # afterwards so that YOLO runs properly
    path = Path.cwd()
    if path.parts[-1] != "labeled-data":
        labeled_data_path = path / "labeled-data"
        os.chdir(labeled_data_path)
    else:
        path = path.parent
    autosplit(path=".", weights=(0.8, 0.12, 0.08))
    os.chdir(path)


# redo_autosplit()

## 4) Train YOLO Pose

In [ ]:
IMGSZ = 448

print(f"Using device training YOLO pose model: {PRETRAINED_WEIGHTS}")

model = YOLO(PRETRAINED_WEIGHTS)
results = model.train(data=str(DATA_YAML), epochs=300, imgsz=IMGSZ, project=MODEL_OUTPUT)
results

## 5) Validate and Export

In [ ]:
metrics = model.val(data=str(DATA_YAML))
metrics

In [ ]:
best_model = Path("./ml_runs/pose/bee_pose/weights/best.pt")
print("Best model path:", best_model)

# Optional exports:
model.export(format="ncnn")
# model.export(format='torchscript')

## 6) Run Best Model on Arbitrary Image Directory

Use the helper below to run inference with `best.pt` on any folder of images and save annotated output images.

In [ ]:
def run_best_pose_on_image_dir(
    image_dir: str | Path,
    model_path: Path = None,
    output_dir: Path = OUTPUT_DIR,
    conf: float = 0.25,
    imgsz: int = 448,
) -> Path:
    """Run best.pt on all images in a directory and save annotated outputs.

    Args:
        image_dir: Directory containing input images.
        model_path: Path to weights. Defaults to PRETRAINED_WEIGHTS.
        output_dir: Directory to write annotated images. Defaults to OUTPUT_DIR.
        conf: Confidence threshold for predictions.
        imgsz: Inference image size.

    Returns:
        Path to the directory containing saved annotated images.
    """
    image_dir = Path(image_dir).expanduser().resolve()
    if model_path is None:
        model_path = PRETRAINED_WEIGHTS
    model_path = Path(model_path).expanduser().resolve()

    output_dir = Path(output_dir).expanduser().resolve()

    if not image_dir.exists() or not image_dir.is_dir():
        raise FileNotFoundError(f"Image directory not found: {image_dir}")
    if not model_path.exists():
        raise FileNotFoundError(f"Model weights not found: {model_path}")

    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    image_files = [p for p in sorted(image_dir.iterdir()) if p.suffix.lower() in image_exts]
    if not image_files:
        raise ValueError(f"No supported images found in: {image_dir}")

    output_dir.parent.mkdir(parents=True, exist_ok=True)

    infer_model = YOLO(model_path)
    infer_model.predict(
        source=str(image_dir),
        conf=conf,
        imgsz=imgsz,
        save=True,
        project=str(output_dir.parent),
        name=output_dir.name,
        exist_ok=True,
    )

    print(f"Annotated images saved to: {output_dir}")
    return output_dir


# Example usage:
run_best_pose_on_image_dir(image_dir=IMAGES_DIR,
                           model_path=PRETRAINED_WEIGHTS)